# 28B Heartbeat: within-filing signals, detect, decide, deduplicate
Every signal compares this period's schedule with the prior fiscal year-end schedule **in the same filing**. Pairing the two is the hard part, and it is yours to design.

In [1]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [2]:
heads = json.load(open(FX / "release_heads.json"))
tick = heartbeat.release_tick(heads["last_seen"], lambda url: heads["heads"].get(url, 404), P["sec"]["base_url"])
print(tick)

{'release': '2026_09', 'url': 'https://www.sec.gov/files/datastandardsinnovation/data/business-development-company-bdc-data-sets/2026_09_bdc.zip', 'published': True}


In [3]:
pairs = signals.pair_exact(PANEL)
print("Exact pairing (same filing, identical identifier):")
print(signals.pairing_report(PANEL, pairs).to_string(index=False))
ex = PANEL[PANEL["name"].str.contains("GOLF") & PANEL["identifier"].str.contains("Cedar Ridge")]
print("\nWhy Golf pairs nothing: the identifier carries the all-in rate, which moved with SOFR")
print(ex[["period_role", "identifier"]].to_string(index=False))

Exact pairing (same filing, identical identifier):
                adsh                         name  debt_holdings  paired  unpaired
0009000011-26-000111    FIXTURE GOLF CAPITAL CORP             10       0        10
0009000012-26-000121 FIXTURE HOTEL SENIOR LENDING              8       8         0
0009000013-26-000131    FIXTURE INDIA CREDIT FUND              7       6         1
0009000014-26-000141  FIXTURE JULIETT LENDING BDC             10       9         1

Why Golf pairs nothing: the identifier carries the all-in rate, which moved with SOFR
period_role                                                                                      identifier
      prior First Lien Senior Secured Loan | Cedar Ridge Dental Partners LLC | S + 5.75% (10.05%) | 09/2030
    current  First Lien Senior Secured Loan | Cedar Ridge Dental Partners LLC | S + 5.75% (9.80%) | 09/2030


In [4]:
found = signals.find_signals(PANEL, M)
print(found[["name", "rule", "identifier", "row", "before", "after"]].to_string(index=False))
if P["data"]["use_fixture"]:
    truth = pd.read_csv(ROOT / P["data"]["truth"], dtype=str)
    sc = evals.signal_scores(found, truth)
    print(f"\nAgainst the planted events: precision {sc['precision']:.2f}, recall {sc['recall']:.2f} "
          f"({sc['tp']} found, {sc['fp']} false, {sc['fn']} missed)")
    print(pd.DataFrame(sc["by_rule"]).T)

                        name     rule                                                        identifier row     before      after
FIXTURE HOTEL SENIOR LENDING EXTENDED               Pinecrest Learning Group Inc., First Lien Term Loan 105 2029-09-30 2031-09-30
FIXTURE HOTEL SENIOR LENDING MARKDOWN            Brightwater Fitness Holdings LLC, First Lien Term Loan 101       99.0       93.5
FIXTURE HOTEL SENIOR LENDING   PIK_UP                   Oakmont Home Services LLC, First Lien Term Loan 104        1.0        3.0
   FIXTURE INDIA CREDIT FUND MARKDOWN Senior Secured First Lien | Westbrook Orthopedics LLC | S + 5.50% 102       96.0       89.5
 FIXTURE JULIETT LENDING BDC MARKDOWN   First Lien Term Loan | Crestline Roofing Holdings LLC | 06/2031 103       97.5       90.0
 FIXTURE JULIETT LENDING BDC   PIK_UP   First Lien Term Loan | Foxglove Specialty Pharma Inc. | 03/2031 106        0.0        2.5

Against the planted events: precision 1.00, recall 0.67 (6 found, 0 false, 3 missed)
    

In [5]:
problems = panel.check_mandates(MANDATES, PANEL)    # a misspelled BDC name would route nothing, silently
assert not problems, problems
held = {m: list(v["positions"]) for m, v in MANDATES.items()}
routed = heartbeat.route(found, held)
seen = set()
early = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW)]
print(f"At {NOW:%H:%M} (quiet hours):", pd.Series([a["action"] for a in early]).value_counts().to_dict(), "| per mandate:",
      pd.Series([a["mandate"] for a in early]).value_counts().to_dict())
again = [a for m, s in routed.items() for a in heartbeat.decide(s, m, seen, NOW.replace(hour=9))]
print("Same signals at 09:00:", again, "(deduplicated)")
recent = json.load(open(FX / "edgar_recent.json"))
for bdc, cik in REG["bdc_cik"].items():
    new = heartbeat.new_filings(recent.get(cik, {"form": [], "items": [], "accessionNumber": [], "acceptanceDateTime": []}), seen=set())
    for m in MANDATES:
        for d in heartbeat.filing_decisions(new, bdc, held[m], m, seen, NOW.replace(hour=9)):
            print(f"  {m}: {d['form']:5s} {d['accn']} {bdc.replace('FIXTURE ', ''):24s} -> {d['action']}")

At 06:00 (quiet hours): {'hold': 9} | per mandate: {'mandate_a': 5, 'mandate_b': 4}
Same signals at 09:00: [] (deduplicated)
  mandate_a: 10-Q  0009000011-26-000111 GOLF CAPITAL CORP        -> alert
  mandate_a: 8-K   0009000011-26-000109 GOLF CAPITAL CORP        -> alert
  mandate_b: 10-Q  0009000011-26-000111 GOLF CAPITAL CORP        -> not_material
  mandate_b: 8-K   0009000011-26-000109 GOLF CAPITAL CORP        -> not_material
  mandate_a: 10-Q  0009000013-26-000131 INDIA CREDIT FUND        -> not_material
  mandate_a: 8-K   0009000013-26-000130 INDIA CREDIT FUND        -> not_material
  mandate_b: 10-Q  0009000013-26-000131 INDIA CREDIT FUND        -> alert
  mandate_b: 8-K   0009000013-26-000130 INDIA CREDIT FUND        -> alert


Exact pairing finds 6 of the 9 planted events with no false alarms: precision 1.00, recall 0.67. It misses Golf's two markdowns (the all-in rate in the identifier changed) and Juliett's extension (the maturity in the identifier changed). Two traps punish a careless fix: Golf's Granite Peak has a term loan and a delayed-draw tranche, only one of them marked down, and Hotel's Redwood Valley fell 4.5 points, under the 5-point threshold.

**Your turn.** Write your own pairing function, pass it as `signals.find_signals(panel, m, pair=your_pairing)`, and measure it: precision and recall on the fixture, plus a hand-checked sample of pairs from your real panel. Then build the four DAGs, deliver alerts and the brief, and measure latency, alerts per mandate-week and idle cost (Part 5).

### Part 5: our pairing (`services/signals/pairing.py`)
Filers write the terms that move into the identifier: the all-in rate (it moves with SOFR), the spread, the maturity (it moves on an extension), sometimes "Cash + PIK". Exact matching therefore misses exactly the holdings whose terms changed, which are the holdings the signals are about. `pair(panel)` works inside one filing (`adsh`), in three passes:

1. **exact**: identical identifier (the kit's baseline);
2. **normalized**: identical once rates, spreads, dates and rate vocabulary are stripped, and only where that key is unique on both sides;
3. **fuzzy**: what is left, by token overlap of the normalized identifiers (Jaccard ≥ 0.8), best pairs first, one-to-one, and **never across tranches**: the words that tell a term loan from its delayed-draw or revolver sibling, or tranche 1 from tranche 2, must agree.

A holding with no counterpart (a new loan, an exit) stays unpaired. No pair is better than a wrong pair, because a wrong pair makes up a markdown, a PIK step-up or an extension.

In [6]:
sys.path.insert(0, str(ROOT))
from services.signals.pairing import pair, normalize
ours = pair(PANEL)
print("Our pairing, by pass:", ours["pair_method"].value_counts().to_dict())
print(signals.pairing_report(PANEL, ours).to_string(index=False))
print("\nWhy Golf pairs now: both schedules normalize to the same key")
for r in ex.itertuples():
    print(f"  {r.period_role:8s} {normalize(r.identifier)}")
found2 = signals.find_signals(PANEL, M, pair=pair)
print()
print(found2[["name", "rule", "identifier", "row", "before", "after"]].to_string(index=False))

Our pairing, by pass: {'exact': 25, 'normalized': 11}
                adsh                         name  debt_holdings  paired  unpaired
0009000011-26-000111    FIXTURE GOLF CAPITAL CORP             10      10         0
0009000012-26-000121 FIXTURE HOTEL SENIOR LENDING              8       8         0
0009000013-26-000131    FIXTURE INDIA CREDIT FUND              7       6         1
0009000014-26-000141  FIXTURE JULIETT LENDING BDC             10      10         0

Why Golf pairs now: both schedules normalize to the same key
  prior    first lien senior secured loan cedar ridge dental partners llc
  current  first lien senior secured loan cedar ridge dental partners llc

                        name     rule                                                                                     identifier row     before      after
   FIXTURE GOLF CAPITAL CORP MARKDOWN First Lien Senior Secured Loan | Cedar Ridge Dental Partners LLC | S + 5.75% (9.80%) | 09/2030 102       98.0       91.5
  

In [7]:
# Precision and recall: the fixture, then the public structural variants (borrowers shuffled, identifier styles
# permuted across BDCs, rows renumbered), so nothing about the fixture's layout can be hard-coded.
import importlib.util
spec = importlib.util.spec_from_file_location("make_fixture", ROOT / "tools" / "make_fixture.py")
mf = importlib.util.module_from_spec(spec); spec.loader.exec_module(mf)
sets = [("fixture", PANEL, truth)]
for s in range(1, 11):
    rows, t = mf.build(s)
    v = pd.DataFrame(rows)
    for c in panel.NUMERIC:
        if c in v: v[c] = pd.to_numeric(v[c], errors="coerce")
    sets.append((f"variant {s}", v, pd.DataFrame(t).astype(str)))
score_rows = []
for label, v, t in sets:
    for name, fn in (("exact", signals.pair_exact), ("ours", pair)):
        sc = evals.signal_scores(signals.find_signals(v, M, pair=fn), t)
        score_rows.append({"panel": label, "pairing": name, "precision": round(sc["precision"], 2),
                           "recall": round(sc["recall"], 2), "tp": sc["tp"], "fp": sc["fp"], "fn": sc["fn"]})
scores = pd.DataFrame(score_rows).pivot(index="panel", columns="pairing", values=["precision", "recall", "fp", "fn"])
print(scores.reindex([l for l, _, _ in sets]).to_string())
sc = evals.signal_scores(found2, truth)
print(f"\nFixture, ours: precision {sc['precision']:.2f}, recall {sc['recall']:.2f} ({sc['tp']} found, {sc['fp']} false, {sc['fn']} missed)")
print(pd.DataFrame(sc["by_rule"]).T)
# The traps: Granite Peak's delayed-draw sibling (unchanged) is paired with itself, not with the marked-down term loan;
# Redwood Valley (4.5 points) stays under the threshold; Thornbury (new, PIK at origination) has no prior and no signal.
gp = ours[ours["identifier"].str.contains("Granite Peak")][["identifier", "identifier_prior", "pair_method"]]
print("\n" + gp.to_string(index=False))
assert not found2["identifier"].str.contains("Delayed Draw|Redwood|Thornbury").any()

           precision      recall         fp         fn     
pairing        exact ours  exact ours exact ours exact ours
panel                                                      
fixture          1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 1        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 2        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 3        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 4        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 5        1.0  1.0   0.78  1.0   0.0  0.0   2.0  0.0
variant 6        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 7        1.0  1.0   0.89  1.0   0.0  0.0   1.0  0.0
variant 8        1.0  1.0   0.78  1.0   0.0  0.0   2.0  0.0
variant 9        1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0
variant 10       1.0  1.0   0.67  1.0   0.0  0.0   3.0  0.0

Fixture, ours: precision 1.00, recall 1.00 (9 found, 0 false, 0 missed)
          found  true  hit
EXTENDED      2     2    2
MARKDOWN      5     5    5
PIK_UP

### Part 5: our pairing on the real panel, and a hand-checked sample
`data/real/panel_2p.parquet` is our Case Study 2B two-period panel: `creditwatch.panel.from_holdings` over both schedules of our canonical holdings (`lantern-lab2b/outputs/canonical/holdings.parquet`), for the 80 filings our CS2B reconciliation admitted (`outputs/panel/holdings.parquet`). The fixture has a ground truth and the real panel does not, so on the real panel we measure **coverage** (debt holdings paired with the prior schedule) and **precision of the pairs** on a fixed-seed sample of 30 pairs from the passes that are not exact (15 normalized, 15 fuzzy). We read each sampled pair against its identifiers, principal and maturity and recorded the verdict in `evals/pairing_handcheck.csv`.

In [8]:
real_path = ROOT / P["data"]["real_panel"]
if not real_path.exists():
    print(f"{real_path.relative_to(ROOT)} not found: export the CS2B two-period panel first (README, Section 4)")
else:
    REAL = panel.load_panel(real_path)
    print(f"real panel: {len(REAL):,} rows, {REAL['adsh'].nunique()} filings | validate:", panel.validate(REAL))
    rp = pair(REAL)
    cov = pd.DataFrame({
        name: signals.pairing_report(REAL, pr)[["debt_holdings", "paired", "unpaired"]].sum()
        for name, pr in (("exact", signals.pair_exact(REAL)), ("ours", rp))}).T
    cov["coverage"] = (cov["paired"] / cov["debt_holdings"]).round(3)
    print(cov.to_string())
    print("our pairs by pass:", rp["pair_method"].value_counts().to_dict())
    rd = rp[rp["principal"] > 0]
    sample = pd.concat([rd[rd["pair_method"] == m].sample(15, random_state=28) for m in ("normalized", "fuzzy")])
    hc = pd.read_csv(ROOT / "evals" / "pairing_handcheck.csv", dtype=str)
    sample = sample.astype({"source_rows": str, "source_rows_prior": str}).merge(
        hc, left_on=["adsh", "source_rows", "source_rows_prior"], right_on=["adsh", "row", "prior_row"], how="left")
    sample["verdict"] = sample["verdict"].fillna("unchecked")
    print("\nHand check:", sample.groupby(["pair_method", "verdict"]).size().to_dict())
    for r in sample[sample["verdict"] != "correct"].itertuples():
        print(f"\n  [{r.pair_method}, {r.verdict}] {r.adsh} row {r.source_rows} / prior {r.source_rows_prior}: {r.note}")
        print(f"    current: {r.identifier[:150]}\n    prior:   {r.identifier_prior[:150]}")
    print("\nTwo correct ones, for flavour:")
    for r in sample[sample["verdict"] == "correct"].head(2).itertuples():
        print(f"  [{r.pair_method}] {r.identifier[:120]}\n             {r.identifier_prior[:120]}")

real panel: 52,507 rows, 80 filings | validate: ['36 duplicate holdings: coalesce first', '3 filing(s) lack a prior schedule']


       debt_holdings  paired  unpaired  coverage
exact          19690    7647     12043     0.388
ours           19690   14455      5235     0.734
our pairs by pass: {'exact': 11337, 'normalized': 5561, 'fuzzy': 2870}

Hand check: {('fuzzy', 'correct'): 13, ('fuzzy', 'doubtful'): 2, ('normalized', 'correct'): 15}

  [fuzzy, doubtful] 0001193125-26-344590 row 96076 / prior 96209: principal 44.85M -> 5.47M and the prior row has no rate: may be a sibling tranche
    current: Investment Debt Investments  145.0% United States - 140.3% 1st Lien/Senior Secured Debt - 136.5% Diligent Corporation Industry Professional Services I
    prior:   Investment Debt Investments  158.5% United States - 153.9% 1st Lien/Senior Secured Debt - 149.8% Diligent Corporation Industry Professional Services R

  [fuzzy, doubtful] 0001193125-26-339729 row 152934;152935 / prior 152936;152937: identifier truncated; industry, maturity and principal (5.06M -> 1.02M) differ: may be another tranche
    current: Portfolio

### Part 5: the heartbeat (`services/heartbeat/`, `dags/`)
Four DAGs, each a thin wrapper over one job in `services/heartbeat/jobs.py` (the logic stays importable, so this notebook and pytest run it with the injected clock and the recorded feeds):

| DAG | Schedule (New York) | Job |
|---|---|---|
| `release_watch_dag` | 06:00 daily | next release out? signals with our pairing, routed to the mandates that hold each BDC, alert or hold |
| `edgar_watch_dag` | every 30 minutes | new 10-Q, 10-K or 8-K Item 2.02 from a held BDC: alert or hold; a logged not-material decision otherwise |
| `monthly_brief_dag` | 07:30 weekdays | on a release day, one brief per mandate, to Priya, with everything held overnight |
| `hygiene_dag` | Mondays 08:00 | watchlist entries not reviewed in 30 days; reporting dates in the next 5 business days |

Quiet hours (21:00 to 07:00, from `params.yaml`) hold an alert for the brief. On a day with no brief, the first EDGAR tick after quiet hours delivers what was held. Dedup keys persist in `outputs/heartbeat/state.json`, so nothing alerts twice. Below we replay one week on the schedule, from the fixture morning (Thursday 15 October, 06:00), and write `outputs/heartbeat/`.

In [9]:
import shutil
from datetime import timedelta
from services.heartbeat import brief_job, edgar_job, hygiene_job, release_job
HB = ROOT / "outputs" / "heartbeat"
shutil.rmtree(HB, ignore_errors=True)                 # the replay rebuilds it from scratch
start, end = NOW, NOW + timedelta(days=7)
runs = []                                             # (time, order, job) on each DAG's schedule
t = start.replace(hour=0, minute=0)
while t < end:
    if t >= start:
        if (t.hour, t.minute) == (6, 0):                       runs.append((t, 0, release_job))
        runs.append((t, 1, edgar_job))
        if (t.hour, t.minute) == (7, 30) and t.weekday() < 5:  runs.append((t, 2, brief_job))
        if (t.hour, t.minute) == (8, 0) and t.weekday() == 0:  runs.append((t, 3, hygiene_job))
    t += timedelta(minutes=30)
for t, _, job in sorted(runs, key=lambda x: (x[0], x[1])):
    job(t)
jl = lambda f: [json.loads(x) for x in (HB / f).read_text().splitlines() if x.strip()]
ALERTS, TICKS, DEC, BRIEFS = (pd.DataFrame(jl(f)) for f in ("alerts.jsonl", "ticks.jsonl", "decisions.jsonl", "briefs.jsonl"))
print(f"{len(runs)} scheduled runs from {start:%a %d %b %H:%M} to {end:%a %d %b %H:%M}:", TICKS["job"].value_counts().to_dict())
print("\nFirst runs:")
print(TICKS.drop(columns=["seconds"]).head(4).astype(object).fillna("").to_string(index=False))
print("\nAlerts and held items, per mandate:")
print(ALERTS.groupby(["mandate", "source", "action"]).size().to_string())
print("\nNot-material filing decisions (logged, never alerted):")
print(DEC[DEC["action"] == "not_material"][["mandate", "bdc", "form", "trigger"]].to_string(index=False))

349 scheduled runs from Thu 15 Oct 06:00 to Thu 22 Oct 06:00: {'edgar_watch': 336, 'release_watch': 7, 'monthly_brief': 5, 'hygiene': 1}

First runs:
found  hold           job model_tokens published release requests                        ts not_material briefs
   12  12.0 release_watch            0      True 2026_09        1 2026-10-15T06:00:00-04:00                    
    8   4.0   edgar_watch            0                          4 2026-10-15T06:00:00-04:00          4.0       
    0         edgar_watch            0                          4 2026-10-15T06:30:00-04:00                    
    0         edgar_watch            0                          4 2026-10-15T07:00:00-04:00                    

Alerts and held items, per mandate:
mandate    source             action
mandate_a  edgar_watch_dag    hold      2
           release_watch_dag  hold      8
mandate_b  edgar_watch_dag    hold      2
           release_watch_dag  hold      4

Not-material filing decisions (logged, never al

In [10]:
# The brief Priya gets at 07:30 on the release day (one per mandate), and the hygiene sweep on Monday
for b in BRIEFS.itertuples():
    print(f"--- {b.brief_id} -> {b.to} ({len(b.alert_ids)} held items)")
    print((ROOT / b.path).read_text())
print("--- hygiene,", json.dumps(json.loads(next((HB / "hygiene").glob("*.json")).read_text()), indent=1))

--- br-2026-10-15-mandate_a -> priya (10 held items)
Monthly brief, mandate_a, 2026-10-15 (release 2026_09)

Exposure: FIXTURE GOLF CAPITAL CORP 42m; FIXTURE HOTEL SENIOR LENDING 18m; FIXTURE JULIETT LENDING BDC 35.5m

Coverage (debt holdings paired with the prior schedule): FIXTURE GOLF CAPITAL CORP 10/10; FIXTURE HOTEL SENIOR LENDING 8/8; FIXTURE JULIETT LENDING BDC 10/10

| BDC | Rule | Signal | Citation |
|---|---|---|---|
| FIXTURE GOLF CAPITAL CORP | MARKDOWN | First Lien Senior Secured Loan \| Cedar Ridge Dental Partners LLC \| S + 5.75% (9.80%) \| 09/2030 marked down 98.0 -> 91.5 (-6.5 points). Filing 0009000011-26-000111, current row 102, prior row 502. | 0009000011-26-000111 |
| FIXTURE GOLF CAPITAL CORP | MARKDOWN | First Lien Senior Secured Loan \| Granite Peak Manufacturing LLC \| S + 5.75% (9.80%) \| 09/2030 marked down 98.5 -> 92.0 (-6.5 points). Filing 0009000011-26-000111, current row 106, prior row 506. | 0009000011-26-000111 |
| FIXTURE HOTEL SENIOR LENDING | EXTENDE

In [11]:
# Isolation and routing, on the file the self-check grades
CAN = json.load(open(FX / "canaries.json"))
for a in ALERTS.itertuples():
    assert a.bdc in MANDATES[a.mandate]["positions"], a.alert_id
    o = policy.confidential_terms(MANDATES, a.mandate)
    o["terms"] += [c for m, cs in CAN.items() if m != a.mandate for c in cs]
    text = json.dumps(jl("alerts.jsonl")[a.Index])
    assert not policy.mentions(text, o["names"]) and not policy.leaks(text, o["terms"]), a.alert_id
print(f"{len(ALERTS)} alerts: every one routed to a mandate that holds its BDC, none names another mandate")
r = __import__("subprocess").run([sys.executable, "-m", "creditwatch.check"], cwd=ROOT, capture_output=True, text=True,
                                 env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")})
# (the self-check's isolation line also reads outputs/tasks/dev.jsonl, which Part 8 writes)
print("\n".join(l for l in r.stdout.splitlines() if "pairing" in l or "alert" in l))

16 alerts: every one routed to a mandate that holds its BDC, none names another mandate


  PASS  services/signals/pairing.py pair() runs on the fixture and variants 1-3  [fixture: P 1.00 R 1.00; variant 1: P 1.00 R 1.00; variant 2: P 1.00 R 1.00; variant 3: P 1.00 R 1.00]
  PASS  outputs/heartbeat/alerts.jsonl: one alert per line, all fields
  PASS  every alert goes only to a mandate that holds its BDC
  FAIL  no alert or task output names, codes or canaries another mandate  [missing outputs/tasks/dev.jsonl]


### Part 5: latency, alerts per mandate-week, idle cost
- **Latency** is measured from when the heartbeat sees an event to when Priya has it: an immediate alert is delivered on the tick that sees it, and a held item at the brief (or, on a day with no brief, at the first tick after quiet hours). On top of that comes the schedule bound: a release is seen within 24 hours of the SEC publishing it (the daily 06:00 tick), a filing within 30 minutes. The replay's EDGAR feed is recorded (accepted in August), so acceptance-to-alert in the replay is not a real latency, and we do not report it as one.
- **Alerts per mandate-week** counts every alert and held item in the replayed week. That week holds the release, so it is the busiest week of the month.
- **Idle cost** is what the ticks that find nothing cost. The heartbeat makes no model call, so idle ticks spend no tokens; what remains is requests to the SEC and CPU time.

In [12]:
A = ALERTS.copy()
A["detected"] = pd.to_datetime(A["detected_at"], utc=True)
st = json.loads((HB / "state.json").read_text())
A["delivered"] = pd.to_datetime(A["alert_id"].map(st["delivered"]), utc=True)
A["latency_min"] = (A["delivered"] - A["detected"]).dt.total_seconds() / 60
assert A["delivered"].notna().all(), "every alert and held item reached Priya"
print("Latency, seen -> delivered (minutes):")
print(A.groupby(["source", "action"])["latency_min"].agg(["count", "min", "median", "max"]).to_string())
print("Schedule bound on top: release published -> seen <= 24 h (daily 06:00 tick); filing accepted -> seen <= 30 min")

A["week"] = A["detected"].dt.tz_convert(P["heartbeat"]["timezone"]).dt.strftime("%G-W%V")
weeks = max(1, (end - start).days / 7)
apmw = A.groupby("mandate").size() / weeks
print(f"\nAlerts per mandate-week (replayed week, {weeks:g} week): " + ", ".join(f"{m} {v:g}" for m, v in apmw.items())
      + f" | mean {apmw.mean():g}")
print(A.groupby(["mandate", "rule"]).size().unstack(fill_value=0).to_string())

idle = TICKS[TICKS["found"] == 0]
pr = P["prices"]
print(f"\nIdle ticks this week: {len(idle)} of {len(TICKS)} ({idle['job'].value_counts().to_dict()})")
print(f"Idle cost per week: {int(idle['model_tokens'].sum())} model tokens = "
      f"${econ.step_usd(idle['model_tokens'].sum(), 0, pr['usd_per_mtok_in'], pr['usd_per_mtok_out']):.2f}; "
      f"{int(idle['requests'].sum())} SEC requests (HEAD for the release, one submissions JSON per BDC per EDGAR tick); "
      f"{idle['seconds'].sum():.2f} s CPU")
one_call = econ.step_usd(2000, 300, pr["usd_per_mtok_in"], pr["usd_per_mtok_out"])
print(f"For comparison: asking a model on every tick (2,000 in / 300 out tokens) would cost ${one_call * len(TICKS):.2f} a week "
      f"at the illustrative prices, ${one_call * len(TICKS) * 52:.2f} a year, to learn mostly that nothing happened.")

Latency, seen -> delivered (minutes):
                          count   min  median   max
source            action                           
edgar_watch_dag   hold        4  90.0    90.0  90.0
release_watch_dag hold       12  90.0    90.0  90.0
Schedule bound on top: release published -> seen <= 24 h (daily 06:00 tick); filing accepted -> seen <= 30 min

Alerts per mandate-week (replayed week, 1 week): mandate_a 10, mandate_b 6 | mean 8
rule       EXTENDED  FILING  MARKDOWN  PIK_UP
mandate                                      
mandate_a         2       2         4       2
mandate_b         1       2         2       1

Idle ticks this week: 345 of 349 ({'edgar_watch': 335, 'release_watch': 6, 'monthly_brief': 4})
Idle cost per week: 0 model tokens = $0.00; 1346 SEC requests (HEAD for the release, one submissions JSON per BDC per EDGAR tick); 0.85 s CPU
For comparison: asking a model on every tick (2,000 in / 300 out tokens) would cost $0.45 a week at the illustrative prices, $23.23 a y